# 作業三｜校園活動備品準備助手

政治大學｜112303086｜會計四｜曾尉庭

**選題：題1，自行出題。**

我想做一個活動前能快速算出備品需求的工具。調整出席率、備品組合和庫存後，就能看到各場次的準備量，以及需要補貨的品項。

資料為四場活動的示範報名數，不是真實個資。每筆報名代表一個場次名額，同一人跨場參加會分別計算。

## 功能與本週內容

| 本週功能 | 在這個程式的用途 |
|---|---|
| `append` | 新增第四場活動，逐筆建立人數串列。 |
| `np.array` | 把報名數轉成可計算的陣列。 |
| `reshape` | 整理成「四場次 × 三種身分」及用量計算需要的形狀。 |
| 廣播運算 | 將出席率、每人備品用量套用到各場次。 |
| 陣列條件篩選 | 只留下庫存不足的品項與補貨數量。 |
| `interact` | 用滑桿與選單即時更新結果。 |

也使用函式、迴圈與 f-string 整理輸出。

**操作方式：** 點「執行階段 → 全部執行」，再調整第4段的滑桿與選單。觀看連結不用申請權限；操作互動功能需要登入 Colab 並連接執行階段。

## 1. 建立報名資料

三種身分依序是一般參加者、工作人員、講者。新增第四場後，把12筆人數整理成4列3欄。

**AI 協助：** 資料結構與 `append`、陣列整理程式。

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown
import numpy as np


def registration_table(rows):
    flat = []
    for row in rows:
        for count in row:
            flat.append(count)
    return np.array(flat, dtype=int).reshape(len(rows), 3)


# 示範資料，新增第四場活動。
registrations = [[60,8,2],[45,6,1],[80,10,2]]
registrations.append([35,5,1])
registration = registration_table(registrations)
print('報名表（一般參加者／工作人員／講者）')
print(registration)
print(f'共 {registration.shape[0]} 場活動，{int(registration.sum())} 個場次名額。')


報名表（一般參加者／工作人員／講者）
[[60  8  2]
 [45  6  1]
 [80 10  2]
 [35  5  1]]
共 4 場活動，255 個場次名額。


## 2. 計算需求與缺貨

出席率是使用者設定的假設，各場次、各身分分別向上取整。每個場次名額都按相同備品組合領取；庫存由四場共用，備品用掉後不重複計算。

| 組合 | 資料袋 | 飲用水 | 點心 |
|---|---:|---:|---:|
| 輕量活動 | 0個 | 1瓶 | 0份 |
| 一般活動 | 1個 | 1瓶 | 1份 |
| 半日工作坊 | 1個 | 2瓶 | 1份 |

**AI 協助：** 出席率換算、廣播運算、加總、條件篩選與報表函式。

In [ ]:
def expected_attendance(table, rate):
    # 各場次、各身分分別向上取整，避免備品不足。
    return (table * rate + 99) // 100


def supply_demand(people, per_person):
    totals = people.sum(axis=1)
    return totals.reshape(-1, 1) * np.array(per_person, dtype=int)


def stock_shortage(demand, stock):
    difference = np.array(demand) - np.array(stock)
    mask = difference > 0
    shortage = np.where(mask, difference, 0)
    return mask, shortage

SESSION_NAMES = ['產品交流場', '設計工作坊', '技術分享場', '合作媒合場']
ITEM_NAMES = np.array(['資料袋', '飲用水', '點心'])
PACKAGES = {'輕量活動': [0,1,0], '一般活動': [1,1,1], '半日工作坊': [1,2,1]}


def make_report(table, rate, package, stock):
    people = expected_attendance(table, rate)
    session_totals = people.sum(axis=1)
    needs = supply_demand(people, PACKAGES[package])
    demand = needs.sum(axis=0)
    stock = np.array(stock, dtype=int)
    mask, shortage = stock_shortage(demand, stock)
    lines = [f'出席率 {rate}%｜備品組合：{package}｜估計共 {int(people.sum())} 人',
             '（各場次、各身分分別向上取整；每筆報名視為一個場次名額。）',
             '\n各場次準備量（資料袋／飲用水／點心）']
    for name, total, row in zip(SESSION_NAMES, session_totals, needs):
        lines.append(f'{name}：{int(total)} 人｜{int(row[0])}／{int(row[1])}／{int(row[2])}')
    lines.append('\n全部場次需求與庫存')
    for name, need, available, gap in zip(ITEM_NAMES,demand,stock,shortage):
        lines.append(f'{name}：需 {int(need)}，庫存 {int(available)}，補 {int(gap)}')
    lines.append('\n補貨清單')
    if mask.any():
        for name, gap in zip(ITEM_NAMES[mask],shortage[mask]):
            lines.append(f'{name}：補 {int(gap)}')
    else:
        lines.append('不需補貨，現有庫存足夠。')
    lines.append('各品項單位不同，補貨數量不混加成單一總數。')
    return '\n'.join(lines)


## 3. 預設結果

出席率80%、一般活動，庫存依序是200個資料袋、220瓶水、180份點心。

**AI 協助：** 以 f-string 顯示設定、各場需求與補貨清單。

In [ ]:
print(make_report(registration,80,'一般活動',[200,220,180]))

出席率 80%｜備品組合：一般活動｜估計共 206 人
（各場次、各身分分別向上取整；每筆報名視為一個場次名額。）

各場次準備量（資料袋／飲用水／點心）
產品交流場：57 人｜57／57／57
設計工作坊：42 人｜42／42／42
技術分享場：74 人｜74／74／74
合作媒合場：33 人｜33／33／33

全部場次需求與庫存
資料袋：需 206，庫存 200，補 6
飲用水：需 206，庫存 220，補 0
點心：需 206，庫存 180，補 26

補貨清單
資料袋：補 6
點心：補 26
各品項單位不同，補貨數量不混加成單一總數。


## 4. 互動操作

出席率可調整0～100%，庫存可調整0～600，備品組合有三種。拖動滑桿或更換選單，結果會即時更新；沒有缺貨時顯示「不需補貨」。

**AI 協助：** `interact`、滑桿、選單與即時更新。

In [8]:


def build_interface(table):
    def show_plan(出席率=80, 備品組合='一般活動', 資料袋庫存=200, 飲用水庫存=220, 點心庫存=180):
        print(make_report(table,出席率,備品組合,[資料袋庫存,飲用水庫存,點心庫存]))

    interact(show_plan,
             出席率=IntSlider(value=80,min=0,max=100,step=5,description='出席率'),
             備品組合=Dropdown(options=list(PACKAGES),value='一般活動',description='備品組合'),
             資料袋庫存=IntSlider(value=200,min=0,max=600,step=10,description='資料袋庫存'),
             飲用水庫存=IntSlider(value=220,min=0,max=600,step=10,description='飲用水庫存'),
             點心庫存=IntSlider(value=180,min=0,max=600,step=10,description='點心庫存'))
    return show_plan.widget

planner = build_interface(registration)


interactive(children=(IntSlider(value=80, description='出席率', step=5), Dropdown(description='備品組合', index=1, op…

## 5. 結果檢查

用手算案例確認人數、備品用量及補貨。0%不需備品，100%應回到原報名數；庫存剛好足夠時，不應列入缺貨。

**AI 協助：** 檢查案例與斷言。

In [ ]:
assert registration.shape == (4,3)
assert int(registration.sum()) == 255
people = expected_attendance(registration,80)
assert people.tolist() == [[48,7,2],[36,5,1],[64,8,2],[28,4,1]]
assert int(people.sum()) == 206
assert supply_demand(people,[1,2,1]).sum(axis=0).tolist() == [206,412,206]
mask, shortage = stock_shortage(np.array([206,206,206]),np.array([200,220,180]))
assert mask.tolist() == [True,False,True]
assert shortage.tolist() == [6,0,26]
assert expected_attendance(registration,0).sum() == 0
assert np.array_equal(expected_attendance(registration,100),registration)
assert not stock_shortage(np.array([206,206,206]),np.array([206,206,206]))[0].any()
print('檢查通過：報名表、出席人數、備品需求、補貨與邊界情況均符合預期。')


檢查通過：報名表、出席人數、備品需求、補貨與邊界情況均符合預期。


## 補充

需求量以設定的出席率估計，不是出席預測。程式不會自動下單，各品項單位不同，也不把它們混加成一個補貨總數。

AI 協助範圍已在各段標示，包含程式與檢查案例。

參考：[ipywidgets 互動功能](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html)、[NumPy 廣播運算](https://numpy.org/doc/stable/user/basics.broadcasting.html)。